## Обучение и оценка базовой модели

1. Загрузка очищенного датасета
2. Разделение на train, validation и test
3. Выделение признаков и целевой переменной
4. Обучение логистической регрессии
5. Оценка на validation

In [1]:
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score

In [2]:
df = pd.read_csv("../data/creditcard_clean.csv")

In [3]:
train_end = int(len(df) * 0.7)
validation_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end]
validation_df = df.iloc[train_end:validation_end]
test_df = df.iloc[validation_end:]

In [4]:
X_train = train_df.drop(columns="Class")
y_train = train_df["Class"]

X_validation = validation_df.drop(columns="Class")
y_validation = validation_df["Class"]

X_test = test_df.drop(columns="Class")
y_test = test_df["Class"]

In [5]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(),
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_val_scores = model.predict_proba(X_validation)[:, 1]

In [8]:
results = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    predictions = (y_val_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_validation, predictions).ravel()

    results.append({
        "threshold": threshold,
        "precision": precision_score(y_validation, predictions),
        "recall": recall_score(y_validation, predictions),
        "detected_fraud": tp,
        "missed_fraud": fn,
        "false_alarms": fp,
    })

pd.DataFrame(results)

,threshold,precision,recall,detected_fraud,missed_fraud,false_alarms
0,0.1,0.712121,0.854545,47,8,19
1,0.2,0.745455,0.745455,41,14,14
2,0.3,0.738095,0.563636,31,24,11
3,0.4,0.743590,0.527273,29,26,10
4,0.5,0.777778,0.509091,28,27,8
5,0.6,0.870968,0.490909,27,28,4


## Выбор порога

Для базовой модели выбран порог 0.1.

На validation модель обнаружила 47 из 55 мошеннических операций
при 19 ложных тревогах. Recall составил 85,5%, precision — 71,2%.

Далее оцениваем модель на test, не меняя выбранный порог.

In [ ]:
selected_threshold = 0.1

y_test_scores = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_scores >= selected_threshold).astype(int)

print(confusion_matrix(y_test, y_test_pred))
print("Precision: ", precision_score(y_test, y_test_pred))
print("Recall: ", recall_score(y_test, y_test_pred))

## Итоговая оценка на test

При пороге 0.1 модель обнаружила 33 из 52 мошеннических операций:
- Precision: 66%.
- Recall: 63,5%.
- Ложные тревоги: 17.
- Пропущенные мошеннические операции: 19.

Качество ниже, чем на validation. Порог по результатам test не менялся.

In [ ]:
joblib.dump(
    {
        "model": model,
        "threshold": selected_threshold,
    },
    "../artifacts/baseline_model.joblib",)